### Lagos Rent Predictor — Model Training

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-Learn 
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Regressors
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import optuna

import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'catboost'

#### Data Loading & Feature Setup

In [5]:
df = pd.read_csv('../data/processed/properties_engineered.csv')
print(f"Loaded {len(df):,} engineered listings")
df.head()

Loaded 15,136 engineered listings


,bedroom_cat,proptype_x_beds,toilets,location_tier,premium_location,luxury_score,has_bq,is_large_premium,amenity_count,property_tier,toilet_bed_ratio,has_pool,has_gym,is_island,price_log,price_annual
0,3,3.0,4.0,1,1,11,0,0,3,1,1.333333,1,1,1,17.504390,40000000.0
1,2,2.0,3.0,2,0,13,0,0,3,1,1.500000,1,1,1,17.909855,60000000.0
2,3,6.0,4.0,2,0,0,0,0,0,2,1.333333,0,0,1,17.034386,25000000.0
3,3,3.0,4.0,2,0,10,1,0,5,1,1.333333,0,0,1,17.034386,25000000.0
4,4,8.0,5.0,2,0,0,0,1,0,2,1.250000,0,0,1,16.118096,10000000.0


In [6]:
# Separate Features (X) and Target (y)
X = df.drop(columns=['price_annual', 'price_log'])
y = df['price_log']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Validation set:  {X_valid.shape[0]:,} samples")

Training set: 12,108 samples
Validation set:  3,028 samples


#### Base Model Evaluation

In [ ]:
models = {
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(random_state=42, n_jobs=-1),
    "LightGBM": LGBMRegressor(random_state=42, n_jobs=-1),
    "CatBoost": CatBoostRegressor(random_state=42, verbose=0) 
}

preprocessor = StandardScaler()
results = []
pipelines = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('scaler', preprocessor),
        ('regressor', model)
    ])
    pipelines[name] = pipeline
    
    # 5-Fold Cross Validation 
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, 
                                scoring='neg_mean_absolute_error', n_jobs=-1)
    
    mae = -cv_scores.mean()
    results.append({'Model': name, 'Log_MAE': mae})

results_df = pd.DataFrame(results).sort_values(by='Log_MAE')
display(results_df)

#### Hyperparameter Tuning using GridSearchCV

In [ ]:
rf_pipeline = Pipeline(steps=[
    ('scaler', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42, n_jobs=-1))
])

param_grid = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__max_features': ['sqrt', 'log2', 1.0],
    'regressor__max_depth': [None, 10, 20]
}

grid_search = GridSearchCV(rf_pipeline, param_grid, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_
print("Best RF parameters:", grid_search.best_params_)

#### Hyperparameter Tuning using Optuna

In [ ]:
def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'verbosity': 0,
        'random_state': 42,
        'n_jobs': -1
    }
    
    pipeline = Pipeline(steps=[
        ('scaler', preprocessor),
        ('regressor', XGBRegressor(**param))
    ])
    
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
    mean_mae = -cv_scores.mean()
    return mean_mae

print("Starting Optuna for XGBoost...")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)
print('Best XGB parameters:', study.best_params)

best_xgb = Pipeline([
    ('scaler', preprocessor),
    ('regressor', XGBRegressor(**study.best_params, random_state=42, n_jobs=-1))
])
best_xgb.fit(X_train, y_train)

#### Ensemble: The Voting Regressor

In [ ]:
print("Building the final Ensemble (Voting Regressor)...")
# Get the base models (normally you use the 'best_xgb' or 'best_rf' from above)
model1 = best_rf.named_steps['regressor']
model2 = best_xgb.named_steps['regressor']
model3 = CatBoostRegressor(random_state=42, verbose=0)

voting_model = VotingRegressor(estimators=[
    ('rf', model1),
    ('xgb', model2),
    ('cat', model3)
])

final_pipeline = Pipeline([
    ('scaler', preprocessor),
    ('ensemble', voting_model)
])

final_pipeline.fit(X_train, y_train)
print("Ensemble trained!")

## 7. Final Evaluation on Test Set
Evaluate our ultimate Ensemble model against the unseen 20% validation set converting Log Price back into pure Naira.

In [ ]:
print("Evaluation on Unseen Test Set:")

# 1. Predict (Note: Predictions are in Log space)
log_preds = final_pipeline.predict(X_test)

# 2. Convert back to absolute Naira (# np.expm1 converts log(x) back to x)
y_test_naira = np.expm1(y_test)
preds_naira = np.expm1(log_preds)

# 3. Metrics
mae = mean_absolute_error(y_test_naira, preds_naira)
rmse = np.sqrt(mean_squared_error(y_test_naira, preds_naira))
r2 = r2_score(y_test_naira, preds_naira)

print(f"Final Model Performance (NGN):")
print(f"MAE:  ₦{mae:,.2f}")
print(f"RMSE: ₦{rmse:,.2f}")
print(f"R^2:  {r2:.4f}")

## 8. Save Model to Disk

In [ ]:
"""
import os
os.makedirs('../artifacts', exist_ok=True)
joblib.dump(final_pipeline, '../artifacts/model.pkl')
print("Model fully trained and saved to artifacts/model.pkl!")
"""